# 🎯 MiniPlaces Scene Recognition - Complete Experiment Notebook

## BMI/CS 771 Assignment 2

This notebook provides a complete pipeline for training and evaluating CNN models on the MiniPlaces dataset.

### Instructions:
1. **Run cells in order** - Each cell depends on previous ones
2. **Enable GPU first** - Go to Runtime → Change runtime type → GPU
3. **Upload assignment2_release.zip** when prompted

---

# Part 1: Environment Setup

In [ ]:
#@title 1.1 Check GPU (MUST BE ENABLED!)
import torch
import sys
import os

print("=" * 60)
print("SYSTEM CHECK")
print("=" * 60)
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"GPU Memory: {gpu_mem:.1f} GB")
    print("\n✅ GPU is ready!")
else:
    print("\n" + "!" * 60)
    print("❌ NO GPU DETECTED!")
    print("Go to: Runtime → Change runtime type → GPU")
    print("Then restart and run this cell again.")
    print("!" * 60)

In [ ]:
#@title 1.2 Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create working directory
SAVE_DIR = '/content/drive/MyDrive/CS771_Assignment2'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"\n✅ Results will be saved to: {SAVE_DIR}")

In [ ]:
#@title 1.3 Upload and Extract Assignment Files
import zipfile
from google.colab import files

os.chdir('/content')

# Check if already extracted
if os.path.exists('/content/assignment2_release/code/train_cls.py'):
    print("✅ Assignment files already extracted!")
else:
    print("📤 Please upload assignment2_release.zip")
    uploaded = files.upload()
    
    for filename in uploaded.keys():
        if filename.endswith('.zip'):
            print(f"\n📦 Extracting {filename}...")
            with zipfile.ZipFile(filename, 'r') as zip_ref:
                zip_ref.extractall('/content/')
            os.remove(filename)
            print("✅ Extraction complete!")

# Verify structure
print("\n📁 Assignment structure:")
!ls -la /content/assignment2_release/

In [ ]:
#@title 1.4 Install Dependencies
# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

!pip install tensorboard pyyaml opencv-python-headless -q
print("✅ Dependencies installed!")

---
# Part 2: Download MiniPlaces Dataset

In [ ]:
#@title 2.1 Download MiniPlaces Dataset (This may take a few minutes)
import os

# Change to assignment root directory
os.chdir('/content/assignment2_release')

# Create data directory
!mkdir -p data

# Change to data directory for download
%cd data

print("📥 Downloading MiniPlaces dataset...")
print("   This may take 5-10 minutes depending on network speed.\n")

# Download the main dataset archive from Google Drive using gdown
!gdown 16GYHdSWS3iMYwMPv5FpeDZN2rH7PR0F2

# Download the label files from the MiniPlaces GitHub repository
!wget -q https://raw.githubusercontent.com/CSAILVision/miniplaces/master/data/train.txt
!wget -q https://raw.githubusercontent.com/CSAILVision/miniplaces/master/data/val.txt

print("\n📦 Extracting dataset (this may take a minute)...")
# Extract the tar.gz archive
!tar -xzf data.tar.gz

# Return to assignment root directory
%cd /content/assignment2_release

# Final verification
print("\n" + "=" * 50)
print("DATASET VERIFICATION")
print("=" * 50)

DATA_DIR = 'data'
required = ['train.txt', 'val.txt', 'images/train', 'images/val']
all_good = True

for name in required:
    path = f'{DATA_DIR}/{name}'
    if os.path.exists(path):
        print(f"✅ {name}")
    else:
        print(f"❌ {name} - MISSING!")
        all_good = False

if all_good:
    print("\n🎉 Dataset is ready!")
    !head -3 data/train.txt
else:
    print("\n❌ Dataset setup incomplete. Check errors above.")

In [ ]:
#@title 2.2 Manual Upload (Only if automatic download failed)
#@markdown Run this cell only if the download above failed.
#@markdown You can manually download data.tar.gz and upload it here.

MANUAL_UPLOAD = False  #@param {type:"boolean"}

if MANUAL_UPLOAD:
    from google.colab import files
    import os
    
    os.chdir('/content/assignment2_release')
    !mkdir -p data
    %cd data
    
    print("📤 Upload data.tar.gz file...")
    uploaded = files.upload()
    
    for filename in uploaded.keys():
        if 'tar.gz' in filename:
            print(f"\n📦 Extracting {filename}...")
            !tar -xzf {filename}
            print("✅ Dataset extracted!")
    
    # Also need label files
    print("\n📥 Downloading label files...")
    !wget -q https://raw.githubusercontent.com/CSAILVision/miniplaces/master/data/train.txt
    !wget -q https://raw.githubusercontent.com/CSAILVision/miniplaces/master/data/val.txt
    
    %cd /content/assignment2_release
    !ls -la data/
else:
    print("ℹ️ Manual upload disabled. Set MANUAL_UPLOAD = True if needed.")

In [ ]:
#@title 2.3 ✅ Verify Dataset
#@markdown Verifies the MiniPlaces dataset is properly downloaded.

import os

os.chdir('/content/assignment2_release')

print("📁 Data directory contents:")
!ls -la data/

print("\n📁 Images directory:")
!ls -la data/images/

# Count the number of training categories
print("\n📊 Dataset statistics:")
num_train_categories = len(os.listdir('data/images/train'))
print(f"   Number of training categories: {num_train_categories}")

# Count lines in label files (number of images)
!echo "   Number of training images: $(wc -l < data/train.txt)"
!echo "   Number of validation images: $(wc -l < data/val.txt)"

# Show a few sample entries from train.txt
print("\n📝 Sample entries from train.txt:")
!head -5 data/train.txt

print("\n" + "=" * 50)
print("🎉 Dataset verification complete!")
print("=" * 50)

---
# Part 3: Setup Improved Models

We'll add these improved architectures:
- **SimpleNetBN**: Baseline + BatchNorm (+2-5% accuracy)
- **ImprovedCNN**: ResNet-style with skip connections (+10-15% accuracy)
- **InceptionNet**: Multi-scale parallel branches

In [ ]:
#@title 3.1 Add Improved Architectures to modeling.py
import os

os.chdir('/content/assignment2_release/code')

# Read current modeling.py
with open('cls_libs/modeling.py', 'r') as f:
    content = f.read()

# Check if already modified
if 'SimpleNetBN' in content:
    print("✅ Improved architectures already added!")
else:
    # New code to insert
    IMPROVED_CODE = '''
################################################################################
# Part I.2: Design your own CNN - IMPROVED ARCHITECTURES
################################################################################


class SimpleNetBN(nn.Module):
    """
    SimpleNet with Batch Normalization.
    Expected improvement: +2-5% accuracy over baseline.
    """
    def __init__(self, conv_op=nn.Conv2d, num_classes=100):
        super(SimpleNetBN, self).__init__()
        self.features = nn.Sequential(
            conv_op(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            conv_op(64, 64, kernel_size=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            conv_op(64, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            conv_op(64, 256, kernel_size=1, bias=False),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            conv_op(256, 128, kernel_size=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            conv_op(128, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            conv_op(128, 512, kernel_size=1, bias=False),
            nn.BatchNorm2d(512), nn.ReLU(inplace=True),
        )
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1.0)
                nn.init.constant_(m.bias, 0.0)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


class ResidualBlock(nn.Module):
    """Residual block with skip connection."""
    def __init__(self, in_ch, out_ch, stride=1, drop_path_rate=0.0):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.drop_path = DropPath(drop_path_rate) if drop_path_rate > 0 else nn.Identity()
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride, bias=False),
                nn.BatchNorm2d(out_ch))

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.drop_path(out)
        return self.relu(out + identity)


class ImprovedCNN(nn.Module):
    """
    ResNet-style CNN with BatchNorm + Skip Connections + DropPath.
    Expected improvement: +10-15% accuracy over baseline.
    """
    def __init__(self, num_classes=100, drop_path_rate=0.1):
        super(ImprovedCNN, self).__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 7, 2, 3, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, 2, 1))
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, 8)]
        self.layer1 = self._make_layer(64, 64, 2, 1, dpr[0:2])
        self.layer2 = self._make_layer(64, 128, 2, 2, dpr[2:4])
        self.layer3 = self._make_layer(128, 256, 2, 2, dpr[4:6])
        self.layer4 = self._make_layer(256, 512, 2, 2, dpr[6:8])
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(512, num_classes)
        self._initialize_weights()

    def _make_layer(self, in_ch, out_ch, n, stride, dpr):
        layers = [ResidualBlock(in_ch, out_ch, stride, dpr[0])]
        for i in range(1, n):
            layers.append(ResidualBlock(out_ch, out_ch, 1, dpr[i]))
        return nn.Sequential(*layers)

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        return self.fc(x)


# ============================================================================
# SELECT YOUR MODEL HERE!
# Options: SimpleNet, SimpleNetBN, ImprovedCNN
# ============================================================================
default_cnn_model = SimpleNet
'''

    # Find and replace Part I.2 section
    import re
    pattern = r'(################################################################################\n# Part I\.2: Design your own CNN[\s\S]*?)(################################################################################\n# Part I\.3)'
    
    new_content = re.sub(pattern, IMPROVED_CODE + '\n\n\\2', content)
    
    with open('cls_libs/modeling.py', 'w') as f:
        f.write(new_content)
    
    print("✅ Added improved architectures to modeling.py!")
    print("\nAvailable models:")
    print("  - SimpleNet (baseline)")
    print("  - SimpleNetBN (+BatchNorm)")
    print("  - ImprovedCNN (ResNet-style)")

In [ ]:
#@title 3.2 Create Experiment Config Files
import yaml
import os

os.chdir('/content/assignment2_release/code')

configs = {
    'exp_baseline': {
        'desc': 'Baseline with default settings',
        'opt': {}
    },
    'exp_strong_aug': {
        'desc': 'Stronger data augmentation',
        'input': {'img_color_jitter': 0.3, 'img_rotation': 25, 'img_min_size': 160}
    },
    'exp_higher_lr': {
        'desc': 'Higher LR with longer warmup',
        'opt': {'learning_rate': 0.2, 'warmup_epochs': 5}
    },
    'exp_best': {
        'desc': 'Best combined settings',
        'input': {'img_color_jitter': 0.25, 'img_rotation': 20, 'img_min_size': 160},
        'opt': {'learning_rate': 0.15, 'weight_decay': 3e-4, 'warmup_epochs': 3, 'epochs': 80}
    }
}

base_config = {
    'output_folder': '../logs',
    'model_type': 'CNN',
    'dataset': {'root_folder': '../data'},
    'loader': {'batch_size': 512, 'num_workers': 4},
    'input': {
        'img_min_size': 144, 'img_crop_size': 128,
        'img_color_jitter': 0.15, 'img_rotation': 15,
        'img_mean': [0.485, 0.456, 0.406],
        'img_std': [0.229, 0.224, 0.225]
    },
    'opt': {
        'type': 'SGD', 'momentum': 0.9, 'weight_decay': 1e-4,
        'learning_rate': 0.1, 'epochs': 60,
        'warmup': True, 'warmup_epochs': 1,
        'schedule_type': 'cosine', 'schedule_steps': [], 'schedule_gamma': 0.1
    }
}

import copy
for name, changes in configs.items():
    cfg = copy.deepcopy(base_config)
    for key, vals in changes.items():
        if key == 'desc':
            continue
        if isinstance(vals, dict):
            cfg[key].update(vals)
        else:
            cfg[key] = vals
    
    with open(f'configs/{name}.yaml', 'w') as f:
        f.write(f"# {changes.get('desc', name)}\n")
        yaml.dump(cfg, f, default_flow_style=False)
    print(f"✅ Created configs/{name}.yaml")

print("\n📁 All config files:")
!ls configs/*.yaml

---
# Part 4: Run Training Experiments

In [ ]:
#@title 4.1 Select Model Architecture
#@markdown Choose which model to train:

MODEL_CHOICE = "SimpleNet" #@param ["SimpleNet", "SimpleNetBN", "ImprovedCNN"]

import os
os.chdir('/content/assignment2_release/code')

# Read and modify modeling.py
with open('cls_libs/modeling.py', 'r') as f:
    content = f.read()

# Update default model
import re
content = re.sub(
    r'default_cnn_model = \w+',
    f'default_cnn_model = {MODEL_CHOICE}',
    content
)

with open('cls_libs/modeling.py', 'w') as f:
    f.write(content)

print(f"✅ Model set to: {MODEL_CHOICE}")
print(f"\nExpected accuracy:")
print(f"  SimpleNet:    ~35-40%")
print(f"  SimpleNetBN:  ~42-47% (+BatchNorm)")
print(f"  ImprovedCNN:  ~50-55% (+Skip Connections +DropPath)")

In [ ]:
#@title 4.2 Run Training
#@markdown Select experiment configuration:

EXPERIMENT = "exp_baseline" #@param ["exp_baseline", "exp_strong_aug", "exp_higher_lr", "exp_best", "cls_miniplaces_cnn"]
EXPERIMENT_NAME = "baseline" #@param {type:"string"}
PRINT_FREQ = 50 #@param {type:"integer"}
CHECKPOINT_FREQ = 10 #@param {type:"integer"}

import os
os.chdir('/content/assignment2_release/code')

# Suppress TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

config_file = f"configs/{EXPERIMENT}.yaml"

print("=" * 60)
print(f"TRAINING: {EXPERIMENT_NAME}")
print(f"Config: {config_file}")
print(f"Model: {MODEL_CHOICE}")
print("=" * 60 + "\n")

!python train_cls.py {config_file} --output {EXPERIMENT_NAME} --print-freq {PRINT_FREQ} --ckpt-freq {CHECKPOINT_FREQ}

print("\n" + "=" * 60)
print(f"✅ TRAINING COMPLETE: {EXPERIMENT_NAME}")
print("=" * 60)

In [ ]:
#@title 4.4 🔄 Resume Training from Checkpoint
#@markdown Use this to continue training from a saved checkpoint.
#@markdown Upload your checkpoint zip first if resuming from a previous session.

import os
import glob

os.chdir('/content/assignment2_release/code')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

#@markdown ---
#@markdown **Settings:**
UPLOAD_CHECKPOINT = False #@param {type:"boolean"}
CHECKPOINT_PATH = "" #@param {type:"string"}
CONFIG_FILE = "configs/exp_baseline.yaml" #@param {type:"string"}
OUTPUT_NAME = "resumed_training" #@param {type:"string"}

# Upload checkpoint if needed
if UPLOAD_CHECKPOINT:
    from google.colab import files
    import zipfile
    
    print("📤 Upload your checkpoint zip file...")
    uploaded = files.upload()
    
    for filename in uploaded.keys():
        if filename.endswith('.zip'):
            print(f"\n📦 Extracting {filename}...")
            with zipfile.ZipFile(filename, 'r') as zip_ref:
                zip_ref.extractall('/content/assignment2_release/')
            os.remove(filename)
            print("✅ Checkpoint extracted!")

# Find available checkpoints
print("\n🔍 Available checkpoints:")
checkpoints = glob.glob('../logs/**/*.pth.tar', recursive=True)
checkpoints = sorted(checkpoints, key=os.path.getmtime, reverse=True)

if not checkpoints:
    print("   ❌ No checkpoints found!")
    print("   Upload a checkpoint zip or run training first.")
else:
    for i, ckpt in enumerate(checkpoints[:10]):  # Show last 10
        size_mb = os.path.getsize(ckpt) / 1e6
        print(f"   {i+1}. {ckpt} ({size_mb:.1f} MB)")
    
    if len(checkpoints) > 10:
        print(f"   ... and {len(checkpoints) - 10} more")

# Resume training
if CHECKPOINT_PATH:
    print(f"\n{'=' * 60}")
    print(f"RESUMING TRAINING")
    print(f"Checkpoint: {CHECKPOINT_PATH}")
    print(f"Config: {CONFIG_FILE}")
    print(f"Output: {OUTPUT_NAME}")
    print(f"{'=' * 60}\n")
    
    !python train_cls.py {CONFIG_FILE} --resume {CHECKPOINT_PATH} --output {OUTPUT_NAME} --print-freq 50
    
    print(f"\n✅ Training resumed and completed!")
else:
    print("\n⚠️ Set CHECKPOINT_PATH to resume training.")
    print("   Example: ../logs/baseline/epoch_030.pth.tar")

In [ ]:
#@title 4.3 Run Multiple Experiments (Batch Mode)
#@markdown Select experiments to run:

RUN_SIMPLENET = True #@param {type:"boolean"}
RUN_SIMPLENET_BN = True #@param {type:"boolean"}
RUN_IMPROVED_CNN = True #@param {type:"boolean"}

import os
os.chdir('/content/assignment2_release/code')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

experiments = []
if RUN_SIMPLENET:
    experiments.append(('SimpleNet', 'baseline_simplenet'))
if RUN_SIMPLENET_BN:
    experiments.append(('SimpleNetBN', 'baseline_simplenet_bn'))
if RUN_IMPROVED_CNN:
    experiments.append(('ImprovedCNN', 'baseline_improved_cnn'))

print(f"Will run {len(experiments)} experiments:\n")
for model, name in experiments:
    print(f"  • {model} → {name}")

results = {}

for i, (model, exp_name) in enumerate(experiments, 1):
    print(f"\n{'#' * 60}")
    print(f"# EXPERIMENT {i}/{len(experiments)}: {model}")
    print(f"{'#' * 60}\n")
    
    # Set model
    with open('cls_libs/modeling.py', 'r') as f:
        content = f.read()
    content = re.sub(r'default_cnn_model = \w+', f'default_cnn_model = {model}', content)
    with open('cls_libs/modeling.py', 'w') as f:
        f.write(content)
    
    # Run training
    !python train_cls.py configs/exp_baseline.yaml --output {exp_name} --print-freq 50 --ckpt-freq 10
    
    print(f"\n✅ Completed: {model}")

print("\n" + "=" * 60)
print("ALL EXPERIMENTS COMPLETE!")
print("=" * 60)

---
# Part 5: View Results

In [ ]:
#@title 5.1 Launch TensorBoard
%load_ext tensorboard
%tensorboard --logdir /content/assignment2_release/logs

In [ ]:
#@title 5.2 List Training Logs
import os
os.chdir('/content/assignment2_release')

print("📁 Training logs:")
!ls -la logs/ 2>/dev/null || echo "No logs yet. Run training first."

print("\n📊 Log details:")
!du -sh logs/*/ 2>/dev/null || echo "No experiment folders yet."

In [ ]:
#@title 5.5 💾 Download ALL Checkpoints (For Resume Training)
#@markdown Downloads all checkpoints so you can resume training later if interrupted.
#@markdown This is useful for:
#@markdown - Resuming training after Colab disconnects
#@markdown - Continuing training on a different machine
#@markdown - Keeping backup of your trained models

import os
import glob
import shutil
import zipfile
from datetime import datetime
from google.colab import files

os.chdir('/content/assignment2_release')

print("=" * 60)
print("CHECKPOINT BACKUP")
print("=" * 60)

# Find all checkpoints
checkpoints = []
checkpoints += glob.glob('logs/**/*.pth.tar', recursive=True)
checkpoints += glob.glob('logs/**/*.pth', recursive=True)

if not checkpoints:
    print("❌ No checkpoints found. Run training first!")
else:
    print(f"\n📁 Found {len(checkpoints)} checkpoint(s):\n")
    
    total_size = 0
    for ckpt in sorted(checkpoints):
        size_mb = os.path.getsize(ckpt) / 1e6
        total_size += size_mb
        print(f"   • {ckpt} ({size_mb:.1f} MB)")
    
    print(f"\n   Total size: {total_size:.1f} MB")
    
    # Create zip file
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    zip_filename = f"checkpoints_backup_{timestamp}.zip"
    zip_path = f'/content/{zip_filename}'
    
    print(f"\n📦 Creating backup: {zip_filename}")
    
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # Add all checkpoints
        for ckpt in checkpoints:
            zipf.write(ckpt, ckpt)
            print(f"   ✅ Added: {ckpt}")
        
        # Also add config files (needed to resume)
        for cfg_file in glob.glob('logs/**/config.txt', recursive=True):
            zipf.write(cfg_file, cfg_file)
            print(f"   ✅ Added: {cfg_file}")
        
        # Add the current modeling.py
        if os.path.exists('code/cls_libs/modeling.py'):
            zipf.write('code/cls_libs/modeling.py', 'code/cls_libs/modeling.py')
            print(f"   ✅ Added: code/cls_libs/modeling.py")
    
    zip_size = os.path.getsize(zip_path) / 1e6
    print(f"\n✅ Backup created: {zip_filename} ({zip_size:.1f} MB)")
    
    # Save to Google Drive
    print(f"\n📁 Saving to Google Drive...")
    try:
        shutil.copy(zip_path, f'{SAVE_DIR}/{zip_filename}')
        print(f"✅ Saved to: {SAVE_DIR}/{zip_filename}")
    except Exception as e:
        print(f"⚠️ Google Drive save failed: {e}")
    
    # Download to local
    print(f"\n💾 Downloading to your computer...")
    try:
        files.download(zip_path)
        print(f"✅ Download started!")
    except Exception as e:
        print(f"⚠️ Auto-download failed: {e}")
        print(f"   Manual: Click folder icon → /content/{zip_filename}")
    
    print("\n" + "=" * 60)
    print("HOW TO RESUME TRAINING")
    print("=" * 60)
    print("""
1. Upload the checkpoint zip to Colab
2. Extract it:
   !unzip checkpoints_backup_*.zip -d /content/assignment2_release/

3. Resume training with --resume flag:
   !python train_cls.py configs/exp_baseline.yaml \\
       --resume logs/YOUR_EXPERIMENT/epoch_XX.pth.tar \\
       --output YOUR_EXPERIMENT_continued
    """)

In [ ]:
#@title 5.3 Save Results to Google Drive AND Download to Local
#@markdown This cell saves all results to Google Drive and creates a zip file for local download.

import shutil
import os
import zipfile
from datetime import datetime
from google.colab import files

os.chdir('/content/assignment2_release')

# Create timestamp for unique naming
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_filename = f"miniplaces_results_{timestamp}.zip"

print("=" * 60)
print("SAVING RESULTS")
print("=" * 60)

# Check if logs exist
if not os.path.exists('logs') or not os.listdir('logs'):
    print("❌ No logs found. Run training first!")
else:
    # ===== SAVE TO GOOGLE DRIVE =====
    print("\n📁 Saving to Google Drive...")
    drive_dst = f'{SAVE_DIR}/logs'
    
    try:
        if os.path.exists(drive_dst):
            shutil.rmtree(drive_dst)
        shutil.copytree('logs', drive_dst)
        print(f"✅ Logs saved to Google Drive: {drive_dst}")
    except Exception as e:
        print(f"⚠️ Google Drive save failed: {e}")
    
    # ===== CREATE ZIP FOR LOCAL DOWNLOAD =====
    print(f"\n📦 Creating zip file: {zip_filename}")
    
    with zipfile.ZipFile(f'/content/{zip_filename}', 'w', zipfile.ZIP_DEFLATED) as zipf:
        # Add logs folder
        for root, dirs, files_list in os.walk('logs'):
            for file in files_list:
                file_path = os.path.join(root, file)
                arcname = file_path  # Keep relative path
                zipf.write(file_path, arcname)
                
        # Add the modified modeling.py
        if os.path.exists('code/cls_libs/modeling.py'):
            zipf.write('code/cls_libs/modeling.py', 'code/cls_libs/modeling.py')
            
        # Add config files
        for cfg_file in os.listdir('code/configs'):
            if cfg_file.endswith('.yaml'):
                zipf.write(f'code/configs/{cfg_file}', f'code/configs/{cfg_file}')
    
    zip_size = os.path.getsize(f'/content/{zip_filename}') / 1e6
    print(f"✅ Zip created: {zip_filename} ({zip_size:.1f} MB)")
    
    # Also save zip to Google Drive
    try:
        shutil.copy(f'/content/{zip_filename}', f'{SAVE_DIR}/{zip_filename}')
        print(f"✅ Zip saved to Google Drive: {SAVE_DIR}/{zip_filename}")
    except Exception as e:
        print(f"⚠️ Could not save zip to Drive: {e}")
    
    # ===== DOWNLOAD TO LOCAL =====
    print(f"\n💾 Downloading to your computer...")
    print("   (A download dialog should appear)")
    
    try:
        files.download(f'/content/{zip_filename}')
        print(f"\n✅ Download started: {zip_filename}")
    except Exception as e:
        print(f"⚠️ Auto-download failed: {e}")
        print(f"   Manual download: Click the folder icon → navigate to /content/{zip_filename}")

print("\n" + "=" * 60)
print("SAVE COMPLETE!")
print("=" * 60)
print(f"\n📍 Results locations:")
print(f"   • Google Drive: {SAVE_DIR}/")
print(f"   • Local download: {zip_filename}")

In [ ]:
#@title 5.4 Download Best Model Checkpoint Only
#@markdown Downloads just the final model checkpoint (smaller file size)

import os
import glob
from google.colab import files

os.chdir('/content/assignment2_release')

print("🔍 Looking for model checkpoints...")

# Find all checkpoint files
checkpoints = glob.glob('logs/**/model_best.pth.tar', recursive=True)
checkpoints += glob.glob('logs/**/epoch_*.pth.tar', recursive=True)

if not checkpoints:
    print("❌ No checkpoints found. Run training first!")
else:
    print(f"\n📁 Found {len(checkpoints)} checkpoint(s):")
    for i, ckpt in enumerate(checkpoints):
        size_mb = os.path.getsize(ckpt) / 1e6
        print(f"   {i+1}. {ckpt} ({size_mb:.1f} MB)")
    
    # Download the most recent/best checkpoint
    latest_ckpt = max(checkpoints, key=os.path.getmtime)
    print(f"\n💾 Downloading: {latest_ckpt}")
    
    try:
        files.download(latest_ckpt)
        print("✅ Download started!")
    except Exception as e:
        print(f"⚠️ Auto-download failed: {e}")
        print(f"   Manual: Click folder icon → navigate to {latest_ckpt}")

---
# 📝 Quick Reference

## Model Comparison

| Model | Features | Expected Accuracy |
|-------|----------|-------------------|
| SimpleNet | Original baseline | ~35-40% |
| SimpleNetBN | + BatchNorm | ~42-47% |
| ImprovedCNN | + BatchNorm + Skip Connections + DropPath | ~50-55% |

## Key Improvements

1. **Batch Normalization**: Stabilizes training, allows higher learning rates
2. **Skip Connections**: Better gradient flow, enables deeper networks
3. **Stochastic Depth (DropPath)**: Regularization for deep networks
4. **He Initialization**: Proper weight scaling for ReLU

## Training Commands

```bash
# From /content/assignment2_release/code directory:
python train_cls.py configs/exp_baseline.yaml --output my_experiment
```